# BÁO CÁO KẾT QUẢ: LỚP 3 - KHAI PHÁ LUẬT KẾT HỢP (MARKET BASKET ANALYSIS)
**Mục tiêu:** Áp dụng phương pháp Khai phá luật kết hợp truyền thống (Apriori) để đi tìm các quy luật Bán chéo đa sản phẩm (Cross-Selling) với giá trị thực tiễn.

**Giới hạn Thách thức:**
* Hệ thống ghi nhận tỷ lệ khách hàng mua 1 sản phẩm cực kỳ cao (96.9%).
* Việc tính toán Tần suất (Support) trên tệp khách hàng mua 1 sản phẩm sẽ làm sai lệch bản chất thuật toán. Do đó, hệ thống sẽ được tự động lọc và **cô lập đúng 1.500 đơn hàng nhiều món** để tiến hành khai phá chuyên sâu.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("✅ Thư viện khai phá Luật kết hợp đã sẵn sàng.")

✅ Thư viện khai phá Luật kết hợp đã sẵn sàng.


## 1. Trích xuất nhóm Dữ liệu Đặc thù (Data Isolation)
Lọc hóa đơn và Biến đổi cấu trúc DataFrame chuẩn bị cho Market Basket.

In [2]:
# 1. Đọc dữ liệu
df = pd.read_csv('../data/orders_with_clusters.csv')

# 2. Nhóm Các Mã Sản Phẩm (MAMH) theo từng Mã Đơn (MADON)
transactions = df.groupby('MADON')['MAMH'].apply(list).reset_index(name='GioHang')

# 3. Lọc vứt đi các đơn chỉ có 1 món (Bảo toàn 1.500 đơn hàng Core)
multi_item_baskets = transactions[transactions['GioHang'].apply(len) > 1]

print(f"🛒 Tổng quy mô hệ thống: {len(transactions)} đơn hàng")
print(f"🔥 Tập dữ liệu cốt lõi (Core Subset): {len(multi_item_baskets)} đơn hàng có >= 2 món")

# Chuyển thành dạng dữ liệu phân nhánh để nạp vào Ma Trận
basket_list = multi_item_baskets['GioHang'].tolist()
print("\nMinh họa cấu trúc Giỏ hàng Đặc thù (Chứa ID Sản Phẩm):")
print(basket_list[:3])

🛒 Tổng quy mô hệ thống: 46047 đơn hàng
🔥 Tập dữ liệu cốt lõi (Core Subset): 1531 đơn hàng có >= 2 món

Minh họa cấu trúc Giỏ hàng Đặc thù (Chứa ID Sản Phẩm):
[[10947, 17011], [1604, 2579], [12517, 6291]]


## 2. Tiền xử lý: Đối mặt với "Ma trận Thưa" (Sparsity Challenge)
Khi trải phẳng 1.500 giỏ hàng này, sự phân tán của chuỗi cung ứng sản phẩm mộc lộ rõ. Tính đa dạng quá cao dẫn đến tỷ lệ trùng lặp sản phẩm giảm mạnh.

In [3]:
# Encode dữ liệu Giỏ hàng thành Cấu trúc Nhị phân
te = TransactionEncoder()
te_ary = te.fit(basket_list).transform(basket_list)
basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"📊 Độ dàn trải giỏ hàng: Tập dữ liệu {basket_encoded.shape[0]} đơn hàng phân tán chéo trên {basket_encoded.shape[1]} Mã Sản Phẩm Khác Nhau!")

📊 Độ dàn trải giỏ hàng: Tập dữ liệu 1531 đơn hàng phân tán chéo trên 2588 Mã Sản Phẩm Khác Nhau!


## 3. Khai phá Quy luật Ngách (Niche Rule Extraction)
Với áp lực từ Ma trận Thưa (2.588 Mã Mặt Hàng), các quy luật kết nối đa sản phẩm mang tính ngách (Niche). Việc thiết lập thông số `min_support = 0.002` (Ngưỡng lặp lại tối thiểu trên 3 giao dịch) cho phép Mô hình Machine Learning quét sâu và bắt được các Điểm bùng phát Bán chéo (Cross-selling points) tiềm năng thay vì bỏ sót chúng.

In [4]:
# Tinh chỉnh tham số: Lọc tìm các Tổ hợp ngách (Micro-Segmentation)
frequent_items = apriori(basket_encoded, min_support=0.002, use_colnames=True)
frequent_items['length'] = frequent_items['itemsets'].apply(lambda x: len(x))

# Lọc Tổ hợp từ 2 sản phẩm trở lên
cap_doi = frequent_items[frequent_items['length'] > 1]
print(f"🔍 Hệ thống dò tìm thành công {len(cap_doi)} Tổ hợp Sản phẩm chéo đáng chú ý.")

# Xuất bản Luật Kinh Doanh
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.1)

if len(rules) > 0:
    print("\n🚀 TOP QUY LUẬT BÁN CHÉO (CROSS-SELL) DÀNH CHO TIẾP THỊ:")
    # Rút gọn báo cáo: Ưu tiên Độ tin cậy (Confidence) và Sức ảnh hưởng (Lift)
    top_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by="lift", ascending=False).head(5)
    print(top_rules)
else:
    print("Hiện tại độ phân tán giỏ hàng quá lớn, chưa chốt được tổ hợp ngách hoàn hảo.")


🔍 Hệ thống dò tìm thành công 9 Tổ hợp Sản phẩm chéo đáng chú ý.

🚀 TOP QUY LUẬT BÁN CHÉO (CROSS-SELL) DÀNH CHO TIẾP THỊ:
   antecedents consequents   support  confidence        lift
0       (1964)     (17597)  0.003919    1.000000  255.166667
1      (17597)      (1964)  0.003919    1.000000  255.166667
13      (6235)     (12441)  0.003266    0.714286  156.224490
12     (12441)      (6235)  0.003266    0.714286  156.224490
15     (19606)      (6460)  0.005225    1.000000  153.100000
